# 03 — Frozen Magi vectors and continuous tail scores
Reuse complete caches first. Cached text boxes are authoritative; match them to DETR object queries without repeating text detection filtering. Old filtered page caches are reused when alignment succeeds and refreshed otherwise. No boxes are dropped.


In [1]:
# Cell purpose: Load full sequences and check the Magi feature fingerprint before doing inference.
# Inputs: sequences.json, local Magi files and extraction/alignment settings.
# Outputs: The reuse status of magi.pt; valid caches avoid model extraction.

from pathlib import Path
import json
from config import CONFIG, ROOT, fingerprint, save_json, require_cache, run_stage, model_files

# Edit config.py once. Restart the kernel after changing it.
# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
CACHE.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print('Existing caches:', [p.name for p in CACHE.iterdir() if p.is_file()])

import numpy as np
import torch
from PIL import Image
from scipy.optimize import linear_sum_assignment
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'sequences.json', '01_prepare_sequence_cache.ipynb')
sequences = json.loads((CACHE / 'sequences.json').read_text(encoding='utf-8'))
MAGI_FEATURES = CACHE / 'magi.pt'
# Fingerprint the inputs/settings so stale feature or sample caches are not silently reused.
signature = fingerprint({'sequences': sequences, 'model': CONFIG['magi_model'],
    'model_files': model_files(CONFIG['magi_model']), 'grayscale': CONFIG['grayscale'], 'text_threshold': CONFIG['text_threshold'],
    'tail_detection_threshold': CONFIG['tail_detection_threshold'],
    'tail_match_threshold': CONFIG['tail_match_threshold'], 'iou': CONFIG['min_bbox_iou'], 'version': 3})
saved = torch.load(MAGI_FEATURES, map_location='cpu', weights_only=True) if MAGI_FEATURES.exists() else None
print('Reuse matching cache:', bool(saved and saved['signature'] == signature))

Existing caches: ['deberta.pt', 'discovery.json', 'sequences.json']
Reuse matching cache: False


In [2]:
# Cell purpose: Define IoU matching, legacy-vector lookup, full-query extraction and local Magi loading.
# Inputs: Cached pixel boxes, images and frozen DETR object/tail query tokens.
# Outputs: Helpers that preserve every original box and require valid one-to-one query alignment.

# Compute every reference/query box overlap in original-page pixel coordinates.
def iou_matrix(a, b):
    a, b = np.asarray(a).reshape(-1, 4), np.asarray(b).reshape(-1, 4)
    intersection = np.maximum(0, np.minimum(a[:, None, 2:], b[None, :, 2:]) - np.maximum(a[:, None, :2], b[None, :, :2])).prod(-1)
    union = (a[:, 2:] - a[:, :2]).prod(-1)[:, None] + (b[:, 2:] - b[:, :2]).prod(-1)[None, :] - intersection
    return intersection / np.maximum(union, 1e-12)

# Index the existing labeled-only cache by sequence/page/text index; it cannot supply missing boxes.
def read_legacy():
    path = CONFIG['legacy_visual_cache']
    if not path or not Path(path).exists():
        return {}
    # Only load your own trusted local legacy file (it may contain non-tensor metadata).
    data = torch.load(path, map_location='cpu', weights_only=False)
    rows = data['samples']
    result = {}
    for row in rows:
        key = (row['sequence_id'], int(row['page_idx']), int(row['text_idx']))
        assert key not in result, f'Duplicate legacy key: {key}'
        result[key] = row
    return result

@torch.inference_mode()
# Extract all object queries; repeated text threshold/NMS filtering could lose a cached box.
def extract_page(model, path):
    from transformers.image_transforms import center_to_corners_format
    with Image.open(path) as image:
        image = image.convert('L').convert('RGB') if CONFIG['grayscale'] else image.convert('RGB')
        array = np.asarray(image)
    height, width = array.shape[:2]
    inputs = model.move_to_device(model.processor.preprocess_inputs_for_detection([array]))
    output = model._get_detection_transformer_output(**inputs)
    logits, normalized = model._get_predicted_bboxes_and_classes(output)
    tokens = model._get_predicted_obj_tokens(output)
    scores, classes = logits.max(-1)
    boxes = center_to_corners_format(normalized) * normalized.new_tensor([width, height, width, height])
    # Cached boxes are authoritative: do not filter their query candidates a second time.
    text_ids = torch.arange(tokens.shape[1], device=boxes.device)
    # Keep detected tail candidates using the configured tail detection threshold.
    tail_ids = model.processor._get_indices_of_tails_to_keep(scores.sigmoid(), classes.long(), boxes, CONFIG['tail_detection_threshold'])[0]
    text_ids = torch.as_tensor(text_ids, dtype=torch.long, device=boxes.device)
    tail_ids = torch.as_tensor(tail_ids, dtype=torch.long, device=boxes.device)
    text_tokens = tokens[0, text_ids]
    # Continuous tail evidence is sigmoid text-tail affinity, not merely a discrete association flag.
    affinities = model._get_text_tail_affinity_matrices(text_obj_tokens_for_this_batch=[text_tokens],
        tail_obj_tokens_for_batch=[tokens[0, tail_ids]], apply_sigmoid=True)[0]
    # Use the strongest detected-tail affinity; no tails means a zero score.
    max_tail = affinities.max(1).values if len(tail_ids) else text_tokens.new_zeros(len(text_ids))
    return boxes[0, text_ids].cpu().numpy(), text_tokens.float().cpu(), max_tail.float().cpu()

# Find a distinct DETR query for every cached box, requiring the configured minimum IoU.
def match_cached_boxes(boxes, detected):
    if len(detected) < len(boxes):
        return None
    ious = iou_matrix([b['bbox'] for b in boxes], detected)
    eligible = ious >= CONFIG['min_bbox_iou']
    # Favor the largest number of valid matches first, then larger IoU among those matches.
    rewards = np.where(eligible, len(boxes) + 1 + ious, -1.)
    rows, cols = linear_sum_assignment(-rewards)
    if len(rows) != len(boxes) or not eligible[rows, cols].all():
        return None
    mapping = np.empty(len(boxes), dtype=int)
    mapping[rows] = cols
    return mapping


# Load local checkpoint weights, skip redundant backbone downloads, and reject missing parameters.
def load_magi(device):
    from transformers import AutoConfig, AutoModel
    config = AutoConfig.from_pretrained(CONFIG['magi_model'], trust_remote_code=True,
                                       local_files_only=True)
    # The local Magi checkpoint already contains the backbone weights.
    config.detection_model_config.use_pretrained_backbone = False
    config.disable_ocr = True
    config.disable_detections = False
    model, info = AutoModel.from_pretrained(CONFIG['magi_model'], config=config,
        trust_remote_code=True, local_files_only=True, output_loading_info=True)
    assert not info['missing_keys'], info['missing_keys']
    assert not info.get('mismatched_keys'), info.get('mismatched_keys')
    model = model.to(device).eval()
    model.requires_grad_(False)
    return model


In [3]:
# Cell purpose: Reuse page features or extract them, align every original box, and assemble Magi features.
# Inputs: Complete sequences, reusable page caches and optional legacy labeled-box vectors.
# Outputs: cache/magi.pt with visual vectors, continuous tail evidence and verified legacy-reuse count.

# Only run the cache-building branch when the stored fingerprint is absent or stale.
if saved is None or saved['signature'] != signature:
    from transformers import AutoModel
    from tqdm.auto import tqdm
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = None
    page_cache = CACHE / 'magi_pages'
    page_cache.mkdir(exist_ok=True)
    legacy = read_legacy()
    vectors, tails = {}, {}
    reused = 0
    for seq in tqdm(sequences, desc='Magi pages'):
        sid = seq['sequence_id']
        vectors[sid], tails[sid] = {}, {}
        for page_idx, page in enumerate(seq['pages']):
            boxes = [b for b in seq['boxes'] if b['page_index'] == page_idx]
            if not boxes:
                continue
            # Page-level fingerprints let interrupted extraction reuse completed pages.
            page_key = fingerprint({'sha256': page['sha256'], 'model': CONFIG['magi_model'],
                'files': model_files(CONFIG['magi_model']), 'gray': CONFIG['grayscale'],
                'text_threshold': CONFIG['text_threshold'],
                'tail_threshold': CONFIG['tail_detection_threshold'], 'version': 1})
            page_file = page_cache / f'{page_key}.pt'
            if page_file.exists():
                cached = torch.load(page_file, map_location='cpu', weights_only=True)
            else:
                if model is None:
                    model = load_magi(device)
                detected, tokens, scores = extract_page(model, page['path'])
                cached = {'boxes': torch.as_tensor(detected), 'tokens': tokens, 'tail_scores': scores,
                          'query_set': 'all_objects_v2'}
                torch.save(cached, page_file)
            detected = cached['boxes'].numpy()
            tokens, scores = cached['tokens'], cached['tail_scores']
            mapping = match_cached_boxes(boxes, detected)
            # An older filtered page cache may be short; retry that page using the full query set.
            if mapping is None and cached.get('query_set') != 'all_objects_v2':
                print(f'{sid} page {page_idx + 1}: retrying alignment with all DETR queries')
                if model is None:
                    model = load_magi(device)
                detected, tokens, scores = extract_page(model, page['path'])
                cached = {'boxes': torch.as_tensor(detected), 'tokens': tokens, 'tail_scores': scores,
                          'query_set': 'all_objects_v2'}
                torch.save(cached, page_file)
                mapping = match_cached_boxes(boxes, detected)
            # Stop on unresolved alignment rather than dropping a box or inventing a visual feature.
            assert mapping is not None, (
                f'{sid} page {page_idx + 1}: cached boxes do not align even with all DETR queries; '
                'check image/model/preprocessing. No boxes were removed.')
            for row, col in enumerate(mapping):
                box = boxes[row]
                vector = tokens[col].clone()
                old = legacy.get((sid, page_idx, box['text_index']))
                if old is not None and np.allclose(old['bbox'], box['bbox'], atol=0.01):
                    candidate = torch.as_tensor(old['detr_embedding']).float()
                    # Verify the legacy model/preprocessing really produced the same query vector.
                    # Reuse a legacy vector only when it numerically agrees with the newly aligned query vector.
                    if candidate.shape == vector.shape and torch.allclose(candidate, vector, atol=1e-4, rtol=1e-3):
                        vector = candidate.clone()
                        reused += 1
                score = float(scores[col])
                vectors[sid][box['box_id']] = vector
                # Store [continuous maximum affinity, above-threshold flag] for the same original box ID.
                tails[sid][box['box_id']] = torch.tensor([score, float(score > CONFIG['tail_match_threshold'])])
    saved = {'signature': signature, 'vectors': vectors, 'tails': tails, 'legacy_vectors_reused': reused}
    torch.save(saved, MAGI_FEATURES)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Visual sequences:', len(saved['vectors']), '| verified legacy vectors:', saved['legacy_vectors_reused'])

Magi pages:   0%|          | 0/80 [00:00<?, ?it/s]

seq_ee880b9612205860 page 1: retrying alignment with all DETR queries


Some weights of the model checkpoint at C:\MODELS\magiv2 were not used when initializing Magiv2Model: ['ocr_model.decoder.model.decoder.embed_positions.weight', 'ocr_model.decoder.model.decoder.embed_tokens.weight', 'ocr_model.decoder.model.decoder.layernorm_embedding.bias', 'ocr_model.decoder.model.decoder.layernorm_embedding.weight', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.k_proj.bias', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.k_proj.weight', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.out_proj.bias', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.out_proj.weight', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.q_proj.bias', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.q_proj.weight', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.v_proj.bias', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn.v_proj.weight', 'ocr_model.decoder.model.decoder.layers.0.encoder_attn_layer_norm.bias', 'ocr_model.decoder.model.

Visual sequences: 80 | verified legacy vectors: 1485
